In [ ]:
import os
os.environ["ROBOFLOW_API_KEY"] = "PMbRJpHoXPb88qiFWJxl"

In [ ]:
!pip uninstall pillow supervision inference scikit-image -y
!pip install pillow==12.2.0 supervision --upgrade

Found existing installation: pillow 12.2.0
Uninstalling pillow-12.2.0:
  Successfully uninstalled pillow-12.2.0
Found existing installation: supervision 0.28.0
Uninstalling supervision-0.28.0:
  Successfully uninstalled supervision-0.28.0
Found existing installation: inference 1.2.7
Uninstalling inference-1.2.7:
  Successfully uninstalled inference-1.2.7
Found existing installation: scikit-image 0.25.2
Uninstalling scikit-image-0.25.2:
  Successfully uninstalled scikit-image-0.25.2
  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached supervision-0.28.0-py3-none-any.whl.metadata (14 kB)
Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.1 MB)
Using cached supervision-0.28.0-py3-none-any.whl (251 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inference

In [ ]:
!pip install inference

  Using cached inference-1.2.7-py3-none-any.whl.metadata (25 kB)
Using cached inference-1.2.7-py3-none-any.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 54.9 MB/s eta 0:00:00


In [ ]:
# Step 1: Install required libraries
!pip install inference supervision opencv-python gradio pandas matplotlib

# Step 2: Import dependencies
import os
import cv2
import numpy as np
import gradio as gr
import supervision as sv
import pandas as pd
import matplotlib.pyplot as plt
from inference import get_model

# Step 3: Load Roboflow model
model = get_model(model_id="detection-xdxfj/1",
    api_key="PMbRJpHoXPb88qiFWJxl")

# Step 4: Detection function
def detect_and_annotate(image):
    # Convert PIL → OpenCV
    image = np.array(image)
    temp_image_path = "temp_uploaded.jpg"
    cv2.imwrite(temp_image_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

    # Run inference
    results = model.infer(temp_image_path, confidence=0.003)[0]
    detections = sv.Detections.from_inference(results)

    # Count objects
    object_counts = {}
    for label in detections.data['class_name']:
        object_counts[label] = object_counts.get(label, 0) + 1

    # Annotate image
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    annotated_image = box_annotator.annotate(scene=image, detections=detections)
    annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

    # Table data
    if object_counts:
        df = pd.DataFrame(list(object_counts.items()), columns=["Object", "Count"])
    else:
        df = pd.DataFrame([["No objects detected", 0]], columns=["Object", "Count"])

    # Bar chart
    fig, ax = plt.subplots(figsize=(5, 4))
    if object_counts:
        ax.bar(df["Object"], df["Count"])
        ax.set_xlabel("Objects")
        ax.set_ylabel("Count")
        ax.set_title("Detected Objects Count")
        plt.xticks(rotation=30, ha="right")
    else:
        ax.text(0.5, 0.5, "No objects detected", ha="center", va="center", fontsize=12)
        ax.axis("off")

    return annotated_image, df, fig


# Step 5: Gradio Dashboard Layout
with gr.Blocks() as demo:
    gr.Markdown("## 📊 Object Detection Dashboard")
    gr.Markdown("Upload an image → Get annotated results. Click **Show Analysis** to view object counts & chart.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Upload Image")
            detect_btn = gr.Button("🔍 Detect Objects")
        with gr.Column():
            output_img = gr.Image(type="numpy", label="Annotated Image")
            analysis_btn = gr.Button("📑 Show Analysis")
            output_table = gr.Dataframe(headers=["Object", "Count"], label="Detected Objects Count", visible=False)
            output_plot = gr.Plot(label="Object Count Chart", visible=False)

    # Linking detection
    detect_btn.click(detect_and_annotate, inputs=input_img, outputs=[output_img, output_table, output_plot])

    # Show analysis button → make hidden outputs visible
    def show_analysis(df, fig):
        return gr.update(visible=True), gr.update(visible=True)

    analysis_btn.click(show_analysis, inputs=[output_table, output_plot], outputs=[output_table, output_plot])

# Step 6: Launch
demo.launch(share=True)


ModelDependencyMissing: Your `inference` configuration does not support SAM3 model. Install SAM3 dependencies and set CORE_MODEL_SAM3_ENABLED to True.
ModelDependencyMissing: Your `inference` configuration does not support Gaze Detection model. Use pip install 'inference[gaze]' to install missing requirements.To suppress this warning, set CORE_MODEL_GAZE_ENABLED to False.
[transformers] `DepthProImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `DepthProImageProcessor` instead.
ModelDependencyMissing: Your `inference` configuration does not support YoloWorld model. Use pip install 'inference[yolo-world]' to install missing requirements.To suppress this warning, set CORE_MODEL_YOLO_WORLD_ENABLED to False.


Output()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd91035c8e3e5ea3b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
